# Serve plate.zarr for remote viewing

Runs *inside* an HPC on-demand Jupyter session (the same conda env your PBS conversion
jobs use, with `blimp` fully installed) -- starts a background HTTP server for an
existing `plate.zarr` on cluster scratch, so a local napari (via
`view_remote_plate_in_napari.ipynb`) can stream it live, without copying the whole
store down first.

Set `PLATE_PATH` below, run the next cell, then follow the printed `ssh` command(s)
from your own laptop -- it tries a direct multi-hop tunnel first, with a
reverse-tunnel-via-login-node fallback if direct SSH to this session's compute node
is blocked. Keep this notebook's kernel running for the length of your viewing
session -- the server stops the moment this kernel does.

In [1]:
from blimp.ome_ngff.plate import serve_plate_over_http

/srv/scratch/z3532965/anaconda3/envs/blimp-dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PLATE_PATH = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/OME-NGFF/20260626_POLR2A_heterogeneity_update.zarr"

# login_alias defaults to "kdm" -- pass your own ssh config Host alias for katana's
# login node if you use a different one, e.g. serve_plate_over_http(PLATE_PATH, login_alias="my-alias").
#
# log_requests=True records every request's path/timing/connection -- see
# summarize_request_log below. Safe to leave on for a normal viewing session too
# (it only adds a small in-memory list, no extra network cost).
served = serve_plate_over_http(PLATE_PATH, log_requests=True)

Serving 'http://127.0.0.1:32437/20260626_POLR2A_heterogeneity_update.zarr' (this session's own compute node: 'k240').

From your laptop, try this first (direct multi-hop tunnel):
    ssh -J kdm -L 32437:localhost:32437 k240

If that's refused (direct SSH to the compute node may be blocked), instead run
this from an interactive terminal in this session:
    ssh -N -R 32437:localhost:32437 kdm
and this from your laptop:
    ssh -L 32437:localhost:32437 kdm

Then, in view_remote_plate_in_napari.ipynb:
    PLATE_PATH = "http://localhost:32437/20260626_POLR2A_heterogeneity_update.zarr"



## Checking what a viewing session actually cost

After doing some viewing from `view_remote_plate_in_napari.ipynb` (`add_plate`,
`attach_plate_wide_measurements`, `add_feature_heatmap`, ...), come back to this kernel
and run the cell below -- it breaks down every request served so far by node kind
(image/label/table) and request kind (metadata/chunk/listing), and prints the
requests-per-connection ratio. Run it again later in the same session to see only
what's happened since last time by clearing first (`served.request_log.clear()`).

In [ ]:
from blimp.ome_ngff.plate import summarize_request_log

print("total requests so far:", len(served.request_log))
summarize_request_log(served)

total requests so far: 0


## When you're done

The server is a daemon thread tied to this kernel -- closing/restarting this notebook,
or letting the on-demand session end, shuts it down automatically. Run the cell below
only if you want to stop it early without ending the whole session.

In [ ]:
served.stop()